# A fully spectral kinematic-match model: guided tour

This notebook walks through `SpectralKM` end to end: what the model represents, how to run
one impact, how to read the diagnostics, and where the model is known to be weak. It is
meant to be read alongside `derivations/paper-formulation.tex`, which is the physics ground truth;
section references below point into it.

**No plotting dependency.** Figures are built as SVG strings and handed to Jupyter directly,
so this notebook runs against the package's own `Project.toml` with nothing extra installed.

**The model in one paragraph.** A droplet and a bath are each represented by a truncated
spectral expansion: the bath surface as $\eta=\sum_{m}a_mJ_0(k_mr)$ on Fourier--Bessel modes
of a finite container of radius $b$, the droplet as $\xi=1+\sum_l\beta_lP_l(\cos\theta)$ on
Legendre modes. They interact only through a contact pressure $p$, expanded on shifted
Legendre polynomials over the *moving* contact patch $\theta\in[0,\theta_c]$. Each mode
responds to pressure through a variable-step BDF2 discretisation that makes every state
variable **affine** in its own pressure moment, so one timestep reduces to a small square
nonlinear system in the pressure coefficients alone. The contact angle $\theta_c$ is *not*
an unknown of that system: it is selected outside it as the feasibility boundary
$\theta_c=\inf\{\theta:\text{non-penetration holds}\}$. That nesting is what makes the inner
problem well conditioned --- condition number $O(1)$ and flat in the timestep, against
$\sim10^{18}$ for the joint system.

There are no collocation points anywhere: every contact condition is imposed weakly, as a
Galerkin projection against the pressure basis in a self-adjoint pairing.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using SpectralKM, Printf, SpecialFunctions, Base64

  Activating 

project at `~/Documents/Github/1pkm-drop-onto-bath/julia`


## 1. Parameters, and how the truncations are chosen

`Params` carries the physics (`We`, `Bo`, `Oh`, bath radius `b`, depth `h0`) and the
numerics (`M` bath modes, `L` droplet modes, `N` pressure modes, `nq` quadrature nodes).
The truncations are not interchangeable -- the figure below shows what each one resolves.

`N` deserves a caveat the picture can't carry on its own. The compliance operator mapping
pressure to gap displacement is *compact*, so its singular values decay and the pointwise
pressure profile **never converges** in `N` --- only its low-order moments, which is all the
dynamics ever sees. Raising `N` therefore buys resolution the model cannot use and costs
conditioning. `N=3` is the production value not because it is converged pointwise but
because the integrated quantities are insensitive to it (checked directly, further down).

In [2]:
p = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=3, b=6.0, h0=3.0, nq=40)
@printf("We=%.4f  Bo=%.4f  Oh=%.4f   M=%d L=%d N=%d nq=%d\n", p.We, p.Bo, p.Oh, p.M, p.L, p.N, p.nq)
@printf("wall condition: %s\n", p.wall)
@printf("first five bath wavenumbers k_m: %s\n", join(round.(p.k[1:5], digits=4), ", "))
println()
println("k_0 = 0 is the piston (uniform) mode; it exists only for a free wall.")
println("Its pressure response is exactly zero, so it cannot be driven -- volume is conserved.")

We=1.0958  Bo=0.0170  Oh=0.0060   M=60 L=60 N=3 nq=40
wall condition: free


first five bath wavenumbers k_m: 0.0, 0.6386, 1.1693, 1.6956, 2.2206



k_0 = 0 is the piston (uniform) mode; it exists only for a free wall.
Its pressure response is exactly zero, so it cannot be driven -- volume is conserved.


In [3]:
W_, H_ = 700, 300
io_ = IOBuffer()
print(io_, "<svg xmlns='http://www.w3.org/2000/svg' width='$W_' height='$H_' font-family='sans-serif' font-size='12'>")
print(io_, "<rect width='$W_' height='$H_' fill='white'/>")

bathy_ = 210.0
xs_ = collect(range(40, 660; length=240))
bathpts_ = join(("$(x),$(bathy_ + 3.5*sin(2pi*6*(x-40)/620))" for x in xs_), " ")
print(io_, "<polyline points='$bathpts_' fill='none' stroke='#3070b3' stroke-width='2'/>")

cx_, r_ = 350.0, 78.0
cy_ = bathy_ - r_
thetas_ = collect(range(0, 2pi; length=240))
droppts_ = join(("$(cx_ + (r_+2.5*sin(5*t))*cos(t)),$(cy_ + (r_+2.5*sin(5*t))*sin(t))" for t in thetas_), " ")
print(io_, "<polygon points='$droppts_' fill='#dbe9fb' fill-opacity='0.9' stroke='#1f4e8c' stroke-width='2'/>")

half_ = 0.42
patch_thetas_ = collect(range(pi/2 - half_, pi/2 + half_; length=40))
patchpts_ = join(("$(cx_ + r_*cos(t)),$(cy_ + r_*sin(t))" for t in patch_thetas_), " ")
print(io_, "<polyline points='$patchpts_' fill='none' stroke='#d95f02' stroke-width='5'/>")
for t in range(pi/2 - half_, pi/2 + half_; length=6)
    print(io_, "<circle cx='$(cx_ + r_*cos(t))' cy='$(cy_ + r_*sin(t))' r='2.6' fill='#7a3d00'/>")
end

labels_ = [
    (60, cy_-95, "#1f4e8c", "L: droplet modes (Legendre) -- resolve the contact patch; L &gt;~ pi/theta_c"),
    (60, bathy_+35, "#3070b3", "M: bath modes (Fourier-Bessel) -- resolve the shortest capillary wave; M=60 at b=6"),
    (60, H_-40, "#d95f02", "N: pressure modes on the patch (kept small -- the pointwise profile never converges)"),
    (60, H_-20, "#7a3d00", "nq: Gauss quadrature nodes on the patch (dots) -- nq &gt;= 2(N+L)"),
]
for (x,y,col,txt) in labels_
    print(io_, "<text x='$x' y='$y' font-size='12' fill='$col'>$txt</text>")
end
print(io_, "<text x='$(W_/2)' y='20' text-anchor='middle' font-size='14'>What M, L, N, nq each control</text></svg>")
HTML(String(take!(io_)))

HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' width='700' height='300' font-family='sans-serif' font-size='12'><rect width='700' height='300' fill='white'/><polyline points='40.0,210.0 42.59414225941423,210.54979251770104 45.18828451882845,211.0859340311044 47.78242677824268,211.59511248180988 50.37656903765691,212.06468528739762 52.97071129707113,212.48299324884115 55.56485355648535,212.83965004112594 58.15899581589958,213.12580009920234 60.75313807531381,213.3343384961265 63.34728033472803,213.4600873539517 65.94142259414225,213.49992440719907 68.53556485355648,213.45286052675607 71.1297071129707,213.3200642793338 73.72384937238493,213.10483291268648 76.31799163179916,212.81251048701228 78.91213389121339,212.45035518528152 81.50627615062761,212.02735909709378 84.10041841004184,211.55402495071806 86.69456066945607,211.04210533691975 89.2887029288703,210.5043108994836 91.88284518828452,209.9539947378796 94.47698744769875,209.40482085815594 97.07112970711297,208.8704249042163 99.6652719665272,208.3640755933113 102.25941422594143,207.89834526208932 104.85355648535565,207.48479770334194 107.44769874476988,207.13370104426264 110.0418410041841,206.8537727952715 112.63598326359832,206.65196339968492 115.23012552301255,206.53328365855552 117.82426778242677,206.5006803156168 120.418410041841,206.55496289147914 123.01255230125523,206.69478358373985 125.60669456066945,206.9166707320734 128.20083682008368,207.21511501738584 130.79497907949792,207.58270625476146 133.38912133891213,208.01031738371904 135.98326359832635,208.48733108741413 138.5774058577406,209.0019034139748 141.1715481171548,209.5412578544178 143.76569037656904,210.09200257537435 146.35983263598325,210.6404629299356 148.9539748953975,211.1730209905813 151.5481171548117,211.67645367380163 154.14225941422595,212.1382610609892 156.73640167364016,212.5469767635988 159.3305439330544,212.89245262640242 161.9246861924686,213.16611069983534 164.51882845188285,213.36115622511818 167.11297071129707,213.47274634386673 169.7071129707113,213.49811034323656 172.30125523012552,213.4366184509894 174.89539748953976,213.28979747234428 177.48953974895397,213.06129288035854 180.0836820083682,212.7567783011126 182.67782426778243,212.38381464112334 185.27196652719664,211.9516623547648 187.86610878661088,211.47105151297905 190.4602510460251,210.95391538232658 193.05439330543933,210.41309412944258 195.64853556485355,209.86201600773114 198.2426778242678,209.31436394223329 200.836820083682,208.78373579115748 203.43096234309624,208.28330671956613 206.02510460251045,207.82550206826878 208.6192468619247,207.4216888403823 211.2133891213389,207.0818934657541 213.80753138075315,206.81455285098076 216.40167364016736,206.6263048962951 218.9958158995816,206.52182368065542 221.5899581589958,206.50370340728895 224.18410041841005,206.57239399124845 226.77824267782427,206.72618988830283 229.3723849372385,206.96127244253367 231.96652719665272,207.27180470117275 234.56066945606693,207.65007634248823 237.15481171548117,208.08669511825008 239.7489539748954,208.57082005737828 242.34309623430963,209.0904306404735 244.93723849372384,209.63262526179435 247.53138075313808,210.1839415680575 250.1255230125523,210.73069072024848 252.71966527196653,211.25929727893146 255.31380753138075,211.75663627391856 257.907949790795,212.21035908906939 260.5020920502092,212.60920007070513 263.0962343096234,212.94325624674258 265.69037656903765,213.20423321129596 268.2845188284519,213.3856510695868 270.87866108786613,213.48300532967772 273.4728033472803,213.4938787461896 276.06694560669456,213.41800133899218 278.6610878661088,213.25725709664187 281.255230125523,213.0156371981244 283.8493723849372,212.6991409143772 286.44351464435147,212.31562665014482 289.0376569037657,211.87461682470493 291.6317991631799,211.38706143615417 294.22594142259413,210.86506617980518 296.8200836820084,210.32159187134457 299.4142259414226,209.77013263788686 302.0083682008368,209.22438086723932 304.60251046025104,208.69788723447795 307.1966527196653,208.2

## 2. A minimal SVG toolkit

Enough to draw line plots, grouped bar charts, and a contact-interval timeline, with no
plotting dependency. Single-quoted XML attributes keep the Julia strings free of escapes.

In [4]:
function svgplot(series; width=680, height=300, xlabel="", ylabel="", title="",
                 xlim=nothing, ylim=nothing)
    pad = 52
    xs = vcat((s.x for s in series)...); ys = vcat((s.y for s in series)...)
    x0, x1 = xlim === nothing ? (minimum(xs), maximum(xs)) : xlim
    y0, y1 = ylim === nothing ? (minimum(ys), maximum(ys)) : ylim
    x1 == x0 && (x1 = x0 + 1); y1 == y0 && (y1 = y0 + 1)
    sx = v -> pad + (v - x0) / (x1 - x0) * (width - 1.6pad)
    sy = v -> height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    io = IOBuffer()
    print(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' font-family='sans-serif' font-size='12'>")
    print(io, "<rect width='$width' height='$height' fill='white'/>")
    # axes with ticks
    for (frac) in 0:0.25:1
        xv = x0 + frac * (x1 - x0); yv = y0 + frac * (y1 - y0)
        print(io, "<line x1='$(sx(xv))' y1='$(height-pad)' x2='$(sx(xv))' y2='$(height-pad+5)' stroke='black'/>")
        print(io, "<text x='$(sx(xv))' y='$(height-pad+18)' text-anchor='middle'>$(round(xv,sigdigits=3))</text>")
        print(io, "<line x1='$pad' y1='$(sy(yv))' x2='$(pad-5)' y2='$(sy(yv))' stroke='black'/>")
        print(io, "<text x='$(pad-9)' y='$(sy(yv)+4)' text-anchor='end'>$(round(yv,sigdigits=3))</text>")
    end
    print(io, "<line x1='$pad' y1='$(height-pad)' x2='$(width-0.6pad)' y2='$(height-pad)' stroke='black'/>")
    print(io, "<line x1='$pad' y1='$(height-pad)' x2='$pad' y2='$(pad*0.5)' stroke='black'/>")
    for s in series
        pts = join(("$(sx(s.x[i])),$(sy(s.y[i]))" for i in eachindex(s.x)), " ")
        w = get(s, :width, 1.8)
        print(io, "<polyline points='$pts' fill='none' stroke='$(s.color)' stroke-width='$w'/>")
    end
    for (i, s) in enumerate(series)
        haskey(s, :label) || continue
        print(io, "<line x1='$(width-190)' y1='$(pad*0.5+14i)' x2='$(width-165)' y2='$(pad*0.5+14i)' stroke='$(s.color)' stroke-width='2.4'/>")
        print(io, "<text x='$(width-159)' y='$(pad*0.5+14i+4)'>$(s.label)</text>")
    end
    print(io, "<text x='$(width/2)' y='$(height-8)' text-anchor='middle'>$xlabel</text>")
    print(io, "<text x='14' y='$(height/2)' text-anchor='middle' transform='rotate(-90 14 $(height/2))'>$ylabel</text>")
    print(io, "<text x='$(width/2)' y='18' text-anchor='middle' font-size='14'>$title</text></svg>")
    return HTML(String(take!(io)))
end

struct HTML s::String end
Base.show(io::IO, ::MIME"text/html", h::HTML) = print(io, h.s)

function svgbars(groups; width=640, height=300, ylabel="", title="", ylim=nothing)
    pad = 56
    vals = Float64[]
    for g in groups, s in g.series
        push!(vals, s.value)
    end
    y0, y1 = ylim === nothing ? (min(0.0, minimum(vals)), maximum(vals) * 1.15) : ylim
    y1 == y0 && (y1 = y0 + 1)
    sy(v) = height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    ngroups = length(groups)
    gw = (width - 1.6pad) / ngroups
    io = IOBuffer()
    print(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' font-family='sans-serif' font-size='12'>")
    print(io, "<rect width='$width' height='$height' fill='white'/>")
    for frac in 0:0.25:1
        yv = y0 + frac * (y1 - y0)
        print(io, "<line x1='$pad' y1='$(sy(yv))' x2='$(pad-5)' y2='$(sy(yv))' stroke='black'/>")
        print(io, "<text x='$(pad-9)' y='$(sy(yv)+4)' text-anchor='end'>$(round(yv,sigdigits=3))</text>")
    end
    print(io, "<line x1='$pad' y1='$(sy(y0))' x2='$(width-0.6pad)' y2='$(sy(y0))' stroke='black'/>")
    print(io, "<line x1='$pad' y1='$(height-pad)' x2='$pad' y2='$(pad*0.5)' stroke='black'/>")
    for (gi, g) in enumerate(groups)
        n = length(g.series)
        bw = gw / (n + 1)
        gx0 = pad + (gi - 1) * gw
        for (si, s) in enumerate(g.series)
            x = gx0 + si * bw - bw * 0.4
            h = sy(y0) - sy(s.value)
            print(io, "<rect x='$x' y='$(sy(s.value))' width='$(bw*0.8)' height='$h' fill='$(s.color)'/>")
            print(io, "<text x='$(x+bw*0.4)' y='$(sy(s.value)-6)' text-anchor='middle' font-size='10'>$(round(s.value,sigdigits=3))</text>")
        end
        print(io, "<text x='$(gx0+gw/2)' y='$(height-pad+18)' text-anchor='middle'>$(g.label)</text>")
    end
    if !isempty(groups) && any(haskey(s, :name) for s in groups[1].series)
        for (i, s) in enumerate(groups[1].series)
            haskey(s, :name) || continue
            print(io, "<line x1='$(width-190)' y1='$(pad*0.5+14i)' x2='$(width-165)' y2='$(pad*0.5+14i)' stroke='$(s.color)' stroke-width='6'/>")
            print(io, "<text x='$(width-159)' y='$(pad*0.5+14i+4)'>$(s.name)</text>")
        end
    end
    print(io, "<text x='14' y='$(height/2)' text-anchor='middle' transform='rotate(-90 14 $(height/2))'>$ylabel</text>")
    print(io, "<text x='$(width/2)' y='18' text-anchor='middle' font-size='14'>$title</text></svg>")
    return HTML(String(take!(io)))
end

function svgtimeline(ivs, primary_idx; width=680, height=170, xlabel="tau", title="")
    pad = 52
    t0 = ivs[1].t_start; t1 = ivs[end].t_end
    span = max(t1 - t0, 1e-9)
    sx(t) = pad + (t - t0) / span * (width - 1.6pad)
    laney = height * 0.55
    io = IOBuffer()
    print(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' font-family='sans-serif' font-size='12'>")
    print(io, "<rect width='$width' height='$height' fill='white'/>")
    print(io, "<line x1='$pad' y1='$(laney+22)' x2='$(width-0.6pad)' y2='$(laney+22)' stroke='black'/>")
    for frac in 0:0.25:1
        tv = t0 + frac * span
        x = sx(tv)
        print(io, "<line x1='$x' y1='$(laney+22)' x2='$x' y2='$(laney+27)' stroke='black'/>")
        print(io, "<text x='$x' y='$(laney+40)' text-anchor='middle'>$(round(tv,digits=2))</text>")
    end
    by = laney - 30
    print(io, "<line x1='$(sx(t0))' y1='$by' x2='$(sx(t1))' y2='$by' stroke='#888' stroke-width='1.4'/>")
    print(io, "<line x1='$(sx(t0))' y1='$(by-5)' x2='$(sx(t0))' y2='$(by+5)' stroke='#888'/>")
    print(io, "<line x1='$(sx(t1))' y1='$(by-5)' x2='$(sx(t1))' y2='$(by+5)' stroke='#888'/>")
    print(io, "<text x='$((sx(t0)+sx(t1))/2)' y='$(by-6)' text-anchor='middle' font-size='11' fill='#888'>span (do not use): $(round(t1-t0,digits=4))</text>")
    for (i, iv) in enumerate(ivs)
        x0 = sx(iv.t_start); x1b = sx(iv.t_end)
        w = max(x1b - x0, 2.0)
        col = i == primary_idx ? "#1f77b4" : "#c9c9c9"
        print(io, "<rect x='$x0' y='$(laney-9)' width='$w' height='18' fill='$col'/>")
        if i == primary_idx
            print(io, "<text x='$((x0+x1b)/2)' y='$(laney-14)' text-anchor='middle' font-size='11' fill='#1f77b4'>primary: $(round(iv.duration,digits=4))</text>")
        end
    end
    print(io, "<text x='$(width/2)' y='$(height-6)' text-anchor='middle'>$xlabel</text>")
    print(io, "<text x='$(width/2)' y='18' text-anchor='middle' font-size='14'>$title</text></svg>")
    return HTML(String(take!(io)))
end
println("ready")

ready


## 3. One impact

`run_simulation` returns three index-aligned things: `levels` (the state at every accepted
step), `diag` (one row per **contact** step, carrying $\theta_c$, the force $f$, and the
solver's own diagnostics), and `phases` (a label per level). Note `diag` is shorter than
`levels` and indexes differently --- that is why `phases` exists.

In [5]:
levels, diag, phases = run_simulation(p; t_end=14.0, dt_init=1e-3)
times = [lv.t for lv in levels]
@printf("accepted steps: %d   of which in contact: %d   t_final: %.3f\n",
        length(levels), length(diag), times[end])
@printf("impact speed: %.4f   rebound speed: %.4f\n", -sqrt(p.We), levels[end].com.v)

accepted steps: 743   of which in contact: 206   t_final: 14.000
impact speed: -1.0468   rebound speed: 0.1481


## 4. Contact time: three definitions, one correct

The single most important trap in reading this model's output. A single physical impact does
**not** produce a single contact interval: the stepper detaches and immediately re-attaches
a few times, because the `just_left` guard forces one advancing free-flight step after
contact ends, after which onset is re-detected at the next step. Those re-entries are
separated by a gap of exactly `dt_init`, which is their signature.

So three scalars all answer to "the contact time", and they differ by about 17%:

- `primary_contact_time` --- the **longest** contiguous interval's duration (highlighted
  below). **This is the metric.** It is the one that measures the single physical event
  rather than the chatter around it, and the one to converge in `M`, `L`, `N`. (Longest, not
  first: an under-resolved run can chatter from the very first step, in which case the first
  interval is a few steps long and the longest one is still the real impact.)
- `contact_time` --- the sum over all intervals shown. Includes the chatter. Legitimate for
  a dissipation budget (total time under load), wrong for an impact duration.
- the first-to-last **span**, bracketed above the timeline --- includes chatter *and* any
  free-flight excursion between re-contacts. Never use it.

Look at the timeline: at these settings the physical contact is the long highlighted bar, and
everything after it is a handful of narrow chatter bars clustered near the end. If the
highlighted bar were ever NOT the first one, that itself would be a sign the run is
chattering at onset and under-resolved.

In [6]:
ivs = contact_intervals(times, phases)
primary_idx = argmax(iv.duration for iv in ivs)
display(svgtimeline(ivs, primary_idx;
        title="Contact intervals: one physical impact, several chatter re-entries"))
@printf("primary_contact_time : %.4f   <- the metric\n", primary_contact_time(times, phases))
@printf("contact_time (total) : %.4f\n", contact_time(times, phases))
println()
@printf("coefficient of restitution: %.4f\n", coefficient_of_restitution(times, levels, phases))
@printf("max penetration depth     : %.4f\n", max_penetration_depth(levels, p.L))

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='170' font-family='sans-serif' font-size='12'><rect width='680' height='170' fill='white'/><line x1='52' y1='115.50000000000001' x2='648.8' y2='115.50000000000001' stroke='black'/><line x1='52.0' y1='115.50000000000001' x2='52.0' y2='120.50000000000001' stroke='black'/><text x='52.0' y='133.5' text-anchor='middle'>0.0</text><line x1='201.2' y1='115.50000000000001' x2='201.2' y2='120.50000000000001' stroke='black'/><text x='201.2' y='133.5' text-anchor='middle'>1.07</text><line x1='350.4' y1='115.50000000000001' x2='350.4' y2='120.50000000000001' stroke='black'/><text x='350.4' y='133.5' text-anchor='middle'>2.13</text><line x1='499.59999999999997' y1='115.50000000000001' x2='499.59999999999997' y2='120.50000000000001' stroke='black'/><text x='499.59999999999997' y='133.5' text-anchor='middle'>3.2</text><line x1='648.8' y1='115.50000000000001' x2='648.8' y2='120.50000000000001' stroke='black'/><text x='648.8' y='133.5' text-anchor='middle'>4.26</text><line x1='52.0' y1='63.500000000000014' x2='648.8' y2='63.500000000000014' stroke='#888' stroke-width='1.4'/><line x1='52.0' y1='58.500000000000014' x2='52.0' y2='68.50000000000001' stroke='#888'/><line x1='648.8' y1='58.500000000000014' x2='648.8' y2='68.50000000000001' stroke='#888'/><text x='350.4' y='57.500000000000014' text-anchor='middle' font-size='11' fill='#888'>span (do not use): 4.2634</text><rect x='52.0' y='84.50000000000001' width='508.0327841676001' height='18' fill='#1f77b4'/><text x='306.01639208380004' y='79.50000000000001' text-anchor='middle' font-size='11' fill='#1f77b4'>primary: 3.6293</text><rect x='560.1727667233772' y='84.50000000000001' width='2.0' height='18' fill='#c9c9c9'/><rect x='623.9600954773616' y='84.50000000000001' width='22.397208924328538' height='18' fill='#c9c9c9'/><rect x='646.4972869574673' y='84.50000000000001' width='2.0' height='18' fill='#c9c9c9'/><rect x='648.0020994320707' y='84.50000000000001' width='2.0' height='18' fill='#c9c9c9'/><text x='340.0' y='164' text-anchor='middle'>tau</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Contact intervals: one physical impact, several chatter re-entries</text></svg>")

primary_contact_time : 3.6293   <- the metric
contact_time (total) : 3.8145



coefficient of restitution: 0.2996
max penetration depth     : 0.7797


## 5. The trajectory, and the contact diagnostics

The droplet's south pole against the bath deflection under it, then the two quantities the
closure actually produces: the selected contact angle $\theta_c(\tau)$ and the net force
$f(\tau)=2\int p\,w\,dx$.

In [7]:
south = [lv.com.z - xi_of_theta(lv.drop.beta, 0.0, p.L) for lv in levels]
eta0  = [sum(lv.bath.a[m+1] for m in 0:p.M) for lv in levels]   # eta(r=0), since J_0(0)=1
svgplot([(x=times, y=south, color="#1f77b4", label="droplet south pole"),
         (x=times, y=eta0,  color="#d62728", label="bath surface at r=0")];
        xlabel="tau", ylabel="height", title="Impact trajectory")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>-0.78</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>3.5</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>-0.0292</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>7.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.721</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>10.5</text><line x1='52' y1='89.30000000000001' x2='47' y2='89.30000000000001' stroke='black'/><text x='43' y='93.30000000000001' text-anchor='end'>1.47</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>14.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>2.22</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,193.04002967341046 52.04262857142857,193.10734608763264 52.10657142857143,193.14587066954476 52.202485714285714,193.17180863829867 52.346357142857144,193.43374142448528 52.56216428571429,193.51952530070167 52.885875,193.84194702567174 53.37144107142857,194.2401209748815 54.09979017857143,195.08157973553674 54.952361607142855,195.85743546631576 55.804933035714285,196.60132524090025 56.657504464285715,197.41398663480294 57.510075892857145,198.21389992637683 58.36264732142857,198.99012353759915 59.21521875,199.79225396737542 60.06779017857143,200.61556685461375 60.92036160714285,201.44986906313014 61.77293303571428,202.29649622342285 62.62550446428571,203.14714855255158 63.47807589285714,203.98826291874963 64.33064732142857,204.8191087966057 65.18321875,205.65060939185565 66.03579017857143,206.49085939189538 66.88836160714285,207.33793067304913 67.74093303571429,208.18461831758515 68.59350446428572,209.02495519017882 69.44607589285715,209.8572817737557 70.29864732142858,210.68335336761774 71.15121875,211.50541526134322 72.00379017857144,212.3246205137231 72.85636160714286,213.14058220425022 73.7089330357143,213.95199979155166 74.56150446428572,214.7574305732229 75.41407589285714,215.55590105515222 76.26664732142858,216.3469578418175 77.11921875000002,217.13030698436057 77.97179017857144,217.90568750515814 78.82436160714286,218.67272780318078 79.6769330357143,219.43086588326358 80.52950446428572,220.17960233053623 81.38207589285716,220.91863122544467 82.23464732142858,221.6476944938255 83.08721875,222.36653755188166 83.93979017857144,223.07492561269757 84.79236160714287,223.77257561842615 85.6449330357143,224.45927954562035 86.49750446428573,225.1347661173381 87.35007589285715,225.79887347783324 88.20264732142859,226.45139943459057 89.05521875000002,227.09214218986497 89.90779017857145,227.72084774504117 90.76036160714287,228.33731161015208 91.6129330357143,228.94145611447405 92.46550446428574,229.5331496284844 93.31807589285717,230.11212604923085 94.17064732142859,230.67836553958776 95.02321875000001,231.23185056839145 95.87579017857145,231.7724362770701 96.72836160714289,232.29997604175156 97.58093303571431,232.81432500812886 98.43350446428573,233.31546576225182 99.28607589285716,233.8034254424628 100.13864732142859,234.2782259109369 100.99121875000003,234.73991255756198 101.84379017857145,235.1885622703294 102.69636160714288,235.62421168492418 103.5489330357143,236.04679729081369 104.40150446428574,236.

In [8]:
td = [d.t for d in diag]
svgplot([(x=td, y=[d.theta_c for d in diag], color="#2ca02c", label="theta_c")];
        xlabel="tau", ylabel="theta_c", title="Contact angle: selected, not solved for")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.001</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.00416</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>1.07</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.243</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>2.13</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.481</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>3.2</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.72</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>4.26</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.958</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,238.99088040376466 52.21002309567917,233.6373996417834 52.52505773919792,227.7265677067222 52.99760970447604,220.92085484778042 53.70643765239323,212.09393918981232 54.769679574269006,202.8581582965139 56.36454245708268,190.9013589607486 58.75683678130318,176.69084085486787 61.55714472369206,161.53923182485875 64.35745266608095,148.68782445683047 67.15776060846983,137.05909681774529 69.95806855085871,127.48957714786192 72.7583764932476,119.55615192678272 75.55868443563648,112.84782652328059 78.35899237802536,107.02746743160301 81.15930032041425,101.58171142246462 83.95960826280313,96.3586141828822 86.75991620519201,91.39732519828655 89.56022414758091,86.73619541942332 92.3605320899698,82.43605487283915 95.16084003235869,78.46087835839396 97.96114797474758,74.76533504903102 100.76145591713646,71.3468306876618 103.56176385952534,68.14735664924214 106.36207180191423,65.1285617504397 109.16237974430311,62.355506831462264 111.962687686692,59.78657401573935 114.76299562908089,57.34850048548489 117.56330357146977,55.10349882925274 120.36361151385867,53.002536830065964 123.16391945624754,51.079694686392486 125.96422739863642,49.3118657154474 128.76453534102532,47.70367879861121 131.5648432834142,46.20043664324706 134.3651512258031,44.834441901351596 137.16545916819197,43.60872087646578 139.96576711058088,42.49593907839781 142.76607505296977,41.467840656553705 145.56638299535865,40.58656576881029 148.36669093774753,39.793100830735 151.16699888013642,39.08851624440331 153.9673068225253,38.47376978276398 156.76761476491419,37.9497044275999 159.56792270730307,37.48614230931196 162.36823064969195,37.14544980522194 165.16853859208084,36.83523028567302 167.96884653446972,36.64882592006907 170.7691544768586,36.4933524220975 173.56946241924751,36.39999999999998 176.3697703616364,36.39999999999998 179.17007830402528,36.431131148703685 181.97038624641417,36.55563750261106 184.77069418880305,36.74228760405947 187.57100213119193,36.928773657567234 190.37131007358084,37.23931057381682 193.17161801596973,37.58040080482439 195.9719259583586,37.95189966597917 198.7722339007475,38.415456960745246 201.57254184313638,38.87799569700229 204.37284978552526,39.37028627384677 207.17315772791414,39.8614230446517 209.97346567030303,40.3514087136501 212.77377361269194,40.80969364966887 215.5740815550808,41.23648641446576 218.3743894974697,41.72324927869113 221.17469743985856,42.330276795894065 223.97500538224747,43.11710064704977 226

In [9]:
svgplot([(x=td, y=[d.f for d in diag], color="#9467bd", label="f"),
         (x=[first(td), last(td)], y=[0.0, 0.0], color="#bbbbbb", width=1.0)];
        xlabel="tau", ylabel="f", title="Net contact force")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.001</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>1.07</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.137</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>2.13</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.274</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>3.2</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.41</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>4.26</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.547</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,247.05691192850833 52.21002309567917,226.2393914364839 52.52505773919792,208.90543563293247 52.99760970447604,221.56534434153082 53.70643765239323,204.97045412819267 54.769679574269006,193.34071845572873 56.36454245708268,179.82652416019448 58.75683678130318,161.65765568812193 61.55714472369206,153.32379322777496 64.35745266608095,140.38610543117744 67.15776060846983,133.57708827704158 69.95806855085871,117.84757386136758 72.7583764932476,112.5131649642384 75.55868443563648,100.75836843627417 78.35899237802536,92.43248622783139 81.15930032041425,85.37376266916462 83.95960826280313,78.3386016869292 86.75991620519201,72.25575019026385 89.56022414758091,66.45175317236163 92.3605320899698,61.59651361242442 95.16084003235869,57.32938545871184 97.96114797474758,53.71280504324824 100.76145591713646,50.4152985876423 103.56176385952534,47.673943911064384 106.36207180191423,45.083922312867884 109.16237974430311,43.033148023070595 111.962687686692,41.40397171523014 114.76299562908089,39.80901595217398 117.56330357146977,38.68063434745724 120.36361151385867,37.76915076349525 123.16391945624754,37.15497945161573 125.96422739863642,36.729399964727975 128.76453534102532,36.52611383878224 131.5648432834142,36.39999999999998 134.3651512258031,36.497321318904454 137.16545916819197,36.769992105478536 139.96576711058088,37.13442375219901 142.76607505296977,37.56340755926729 145.56638299535865,38.22368394225188 148.36669093774753,38.892310309206465 151.16699888013642,39.63902177222906 153.9673068225253,40.47330580891466 156.76761476491419,41.38551258002644 159.56792270730307,42.31205108870037 162.36823064969195,43.413803943370965 165.16853859208084,44.42164592274233 167.96884653446972,45.58010643581423 170.7691544768586,46.860780037242876 173.56946241924751,47.81371165172959 176.3697703616364,49.23563895467319 179.17007830402528,50.410593923197354 181.97038624641417,51.8102014673274 184.77069418880305,53.150250352171156 187.57100213119193,54.39448337347528 190.37131007358084,55.843645539906305 193.17161801596973,57.39945706734744 195.9719259583586,58.85730016126075 198.7722339007475,60.36296687736018 201.57254184313638,61.91721094728976 204.37284978552526,63.44189071888093 207.17315772791414,64.66784037647932 209.97346567030303,65.69377719996419 212.77377361269194,66.39601997036445 215.5740815550808,66.80811405031446 218.3743894974697,67.02549970040096 221.17469743985856,67.63806591614537 223.97500538224747,68.726682161860

Two things to notice in the force trace. It is positive almost throughout --- the pressure
pushes the droplet *up*, as it must --- and it can dip transiently negative early without
contact having ended. Loss of contact therefore requires $f\le0$ on **three consecutive**
steps rather than a single sign change: a test that reacts to one negative step sends the
stepper back to free flight, where onset is immediately re-detected and an expensive onset
search is paid again, so a single transient dip should not end contact.

## 6. Interface shapes, and what the pressure does not do

Reconstruct both interfaces at a few instants during contact.

In [10]:
contact_idx = findall(==(InContact), phases)
picks = contact_idx[round.(Int, range(1, length(contact_idx); length=4))]
rg = range(0.0, 3.0; length=240)
cols = ["#08306b", "#2171b5", "#6baed6", "#c6dbef"]
series = NamedTuple[]
for (c, i) in zip(cols, picks)
    push!(series, (x=collect(rg), y=reconstruct_bath(levels[i], collect(rg), p),
                   color=c, label="tau=$(round(times[i],digits=2))"))
end
svgplot(series; xlabel="r", ylabel="eta", title="Bath surface during contact")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>-0.627</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>0.75</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>-0.445</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>1.5</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>-0.264</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>2.25</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>-0.0824</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>3.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.099</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,65.28239887998896 54.49707112970711,65.28225228892117 56.994142259414225,65.28182465523773 59.49121338912134,65.28115117386818 61.98828451882845,65.28028656093909 64.48535564853556,65.2792996027492 66.98242677824268,65.27826639199569 69.47949790794979,65.27726299410679 71.9765690376569,65.27635833796981 74.47364016736401,65.27560808431895 76.97071129707112,65.27505009777653 79.46778242677824,65.27470195163721 81.96485355648535,65.27456065277369 84.46192468619248,65.27460451735391 86.95899581589958,65.27479688743603 89.4560669456067,65.27509118231004 91.9531380753138,65.27543664889086 94.45020920502091,65.27578412595534 96.94728033472803,65.2760911707291 99.44435146443514,65.27632600606259 101.94142259414225,65.27646991581705 104.43849372384938,65.27651792190409 106.93556485355649,65.27647779179338 109.4326359832636,65.27636762318164 111.9297071129707,65.27621240914843 114.42677824267781,65.27604008496286 116.92384937238494,65.27587758741231 119.42092050209205,65.27574741886903 121.91799163179915,65.27566510980512 124.41506276150626,65.27563783093902 126.91213389121339,65.27566424038937 129.4092050209205,65.2757354848807 131.9062761506276,65.27583712888605 134.40334728033474,65.27595167940854 136.90041841004182,65.27606131866972 139.39748953974896,65.27615045670558 141.89456066945607,65.27620776761128 144.39163179916318,65.27622746696679 146.8887029288703,65.27620970871399 149.3857740585774,65.27616010949785 151.8828451882845,65.27608852900121 154.37991631799161,65.27600733009893 156.87698744769875,65.27592940103088 159.37405857740583,65.2758662372548 161.87112970711297,65.27582635345061 164.36820083682005,65.27581423248554 166.8652719665272,65.27582992896035 169.36234309623433,65.27586934420529 171.8594142259414,65.27592509215415 174.35648535564854,65.27598779504743 176.85355648535563,65.27604759497225 179.35062761506273,65.27609564796109 181.84769874476987,65.27612538277555 184.34476987447698,65.27613335261321 186.8418410041841,65.27611957660588 189.3389121338912,65.27608734808734 191.8359832635983,65.27604256602717 194.33305439330545,65.27599271325457 196.83012552301253,65.27594565092281 199.32719665271966,65.27590841741747 201.82426778242677,65.2758862101027 204.32133891213388,65.27588169268287 206.818410041841,65.27589471592722 209.31548117154813,65.27592247402094 211.8125523012552,65.27596005298392 214.30962343096232,65.27600127115429 216.80669456066946,65.27603967261854 219.30376569037657,65

## 7. Sensitivity to the pressure truncation `N`

The pointwise pressure profile does not converge in `N` (the compliance operator is
compact); the question is whether that matters for what gets reported. It runs a few
impacts at different `N` and checks.

In [11]:
Ns = (2, 3, 6)
tcs_N = Float64[]; cors_N = Float64[]; rcs_N = Float64[]
for N in Ns
    pn = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=N, b=6.0, h0=3.0, nq=max(40, 10N))
    lv, dg, ph = run_simulation(pn; t_end=14.0, dt_init=1e-3)
    tt = [l.t for l in lv]
    push!(tcs_N, primary_contact_time(tt, ph))
    push!(cors_N, coefficient_of_restitution(tt, lv, ph))
    push!(rcs_N, sin(maximum(d.theta_c for d in dg)))
end
reltc_N = 100 .* (tcs_N .- tcs_N[end]) ./ tcs_N[end]
relcor_N = 100 .* (cors_N .- cors_N[end]) ./ cors_N[end]
display(svgplot([(x=collect(Ns), y=reltc_N, color="#1f77b4", label="t_cont"),
                  (x=collect(Ns), y=relcor_N, color="#d62728", label="CoR")];
                 xlabel="N", ylabel="% deviation from N=6",
                 title="The integrated metrics barely move with N"))
display(svgplot([(x=collect(Ns), y=rcs_N, color="#2ca02c", label="r_c max")];
                 xlabel="N", ylabel="r_c max",
                 title="...but the contact-radius maximum keeps drifting with N"))

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>2.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>-0.00498</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>3.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>-0.00373</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>4.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>-0.00249</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>5.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>-0.00124</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>6.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.0</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 201.2,36.39999999999998 648.8,36.39999999999998' fill='none' stroke='#1f77b4' stroke-width='1.8'/><polyline points='52.0,248.0 201.2,196.6371818195311 648.8,36.39999999999998' fill='none' stroke='#d62728' stroke-width='1.8'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>t_cont</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>CoR</text><text x='340.0' y='292' text-anchor='middle'>N</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% deviation from N=6</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>The integrated metrics barely move with N</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>2.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.812</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>3.0</text><line x1='52' y1='195.0999999999991' x2='47' y2='195.0999999999991' stroke='black'/><text x='43' y='199.0999999999991' text-anchor='end'>0.814</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>4.0</text><line x1='52' y1='142.20000000000172' x2='47' y2='142.20000000000172' stroke='black'/><text x='43' y='146.20000000000172' text-anchor='end'>0.816</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>5.0</text><line x1='52' y1='89.30000000000086' x2='47' y2='89.30000000000086' stroke='black'/><text x='43' y='93.30000000000086' text-anchor='end'>0.817</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>6.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.819</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 201.2,64.63505776332522 648.8,248.0' fill='none' stroke='#2ca02c' stroke-width='1.8'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='2.4'/><text x='521' y='44.0'>r_c max</text><text x='340.0' y='292' text-anchor='middle'>N</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>r_c max</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>...but the contact-radius maximum keeps drifting with N</text></svg>")

## 8. Watch the impact

The same renderer `scripts/make_video.jl` uses, run here over a downsampled set of frames and
encoded to a short inline clip with `ffmpeg` --- already a dependency of that script, not a
new one. For the full-resolution video, run `scripts/make_video.jl` directly from the shell.

In [12]:
include(joinpath(@__DIR__, "..", "scripts", "make_video.jl"))

# Same fixed-inset-range logic make_video.jl's own main() uses, over the current run.
fmap_v = Dict(round(d.t, digits=12) => d.f for d in diag)
fmax_v = Float64[]; fmin_v = Float64[]; rc_max_v = 1e-9
for lv in levels
    lv.X === nothing && continue
    thc = lv.X[end]; thc <= 0 && continue
    chat = lv.X[1:p.N+1]; xc = cos(thc)
    vs = [pressure_poly_raw(chat, xc, x) for x in range(xc, 1.0; length=110)]
    push!(fmax_v, maximum(vs)); push!(fmin_v, minimum(vs))
    for x in range(xc, 1.0; length=110)
        rc_max_v = max(rc_max_v, forward_map_r(lv.drop.beta, acos(clamp(x, -1, 1)), p.L))
    end
end
pctv(v, q) = (sort(v))[clamp(cld(round(Int, q * length(v)), 1), 1, length(v))]
pmax_v = isempty(fmax_v) ? 1.0 : 1.25 * pctv(fmax_v, 0.90)
pmin_v = isempty(fmin_v) ? 0.0 : min(0.0, 1.25 * pctv(fmin_v, 0.10))

nframes_v = 90
tgrid_v = range(times[1], times[end]; length=nframes_v)
clippath = joinpath(mktempdir(), "clip.mp4")
open(`ffmpeg -y -loglevel error -f image2pipe -vcodec ppm -r 24 -i - -vcodec libx264
      -pix_fmt yuv420p -crf 23 $clippath`, "w") do io
    for t in tgrid_v
        lvl = levels[argmin(abs.(times .- t))]
        f = get(fmap_v, round(lvl.t, digits=12), 0.0)
        c = frame(lvl, p, f, rc_max_v, pmin_v, pmax_v)
        write(io, "P6\n$W $H\n255\n")
        buf = Vector{UInt8}(undef, W * H * 3)
        k = 1
        for i in 1:H, j in 1:W, ch in 1:3
            buf[k] = c.px[i, j, ch]; k += 1
        end
        write(io, buf)
    end
end
b64 = Base64.base64encode(read(clippath))
HTML("""<video width='440' autoplay loop muted playsinline>
<source src='data:video/mp4;base64,$b64' type='video/mp4'></source></video>""")

HTML("<video width='440' autoplay loop muted playsinline>\n<source src='data:video/mp4;base64,AAAAIGZ0eXBpc29tAAACAGlzb21pc28yYXZjMW1wNDEAAAAIZnJlZQAC4s5tZGF0AAACrwYF//+r3EXpvebZSLeWLNgg2SPu73gyNjQgLSBjb3JlIDE2NCByMzEwOCAzMWUxOWY5IC0gSC4yNjQvTVBFRy00IEFWQyBjb2RlYyAtIENvcHlsZWZ0IDIwMDMtMjAyMyAtIGh0dHA6Ly93d3cudmlkZW9sYW4ub3JnL3gyNjQuaHRtbCAtIG9wdGlvbnM6IGNhYmFjPTEgcmVmPTMgZGVibG9jaz0xOjA6MCBhbmFseXNlPTB4MzoweDExMyBtZT1oZXggc3VibWU9NyBwc3k9MSBwc3lfcmQ9MS4wMDowLjAwIG1peGVkX3JlZj0xIG1lX3JhbmdlPTE2IGNocm9tYV9tZT0xIHRyZWxsaXM9MSA4eDhkY3Q9MSBjcW09MCBkZWFkem9uZT0yMSwxMSBmYXN0X3Bza2lwPTEgY2hyb21hX3FwX29mZnNldD0tMiB0aHJlYWRzPTEyIGxvb2thaGVhZF90aHJlYWRzPTIgc2xpY2VkX3RocmVhZHM9MCBucj0wIGRlY2ltYXRlPTEgaW50ZXJsYWNlZD0wIGJsdXJheV9jb21wYXQ9MCBjb25zdHJhaW5lZF9pbnRyYT0wIGJmcmFtZXM9MyBiX3B5cmFtaWQ9MiBiX2FkYXB0PTEgYl9iaWFzPTAgZGlyZWN0PTEgd2VpZ2h0Yj0xIG9wZW5fZ29wPTAgd2VpZ2h0cD0yIGtleWludD0yNTAga2V5aW50X21pbj0yNCBzY2VuZWN1dD00MCBpbnRyYV9yZWZyZXNoPTAgcmNfbG9va2FoZWFkPTQwIHJjPWNyZiBtYnRyZWU9MSBjcmY9MjMuMCBxY29tcD0wLjYwIHFwbWluPTAgcXBtYXg9NjkgcXBzdGVwPTQgaXBfcmF0aW89MS40MCBhcT0xOjEuMDAAgAAAG31liIQAO//+906/AptFhX+B86sID7ziVdCDk5gtVi1K7Gan9fnfOfFumvvDbJXQhdLhpNRb6j30XIUZ29B/aXiE2jfTxh77wIHAr5OD774UKZF2UzjV+19V7j3uC9TIKQpXDg5vOeoISztgHzZ3/P320ao7nEfHhMSMxugNxVCAZCIKBzbRzV9wuXKAAAADAAADAhT9Nbe02waZD+wLB5T1MwyhKdzDYCHOwrdwLP9pXClwBU2DGJMV+dZkiG0bX8CYGamuRQB2+3PVduRbvQfxaoHutPAlCI7i1zUghFfaGABMvDAF9V8g6OoNetnRwe5UXo/QqOmUqkGnsALYYGTlFIl+n+P+rAQtr81SzSzqQRCq3WvXPOB7c0t9apLjBVOHR3vKUuvLrYwHhCHhzoYCk8kNw2UBFDA4Pn5dGmzfYae+yaTJgGo290Yo+QcVmg/qYkMab7wP/RthTYsSphzlOfkzC5Pm7F3zivB0gw3AtGxVcHg/ZchHdyHaWOTC/X+YtTsZ6tDW+WiAmDdbVdvSwh5k/D5uVh/F0uDEk59Kel67m+U000DMqvz64Ul9YUTCXe9ffes7+N+oxEMG5ZrgVE7yFU1kMLXxoWqDtHRwAYDesz7uYnm1et58qt6xlFJEyXGJJ+XYhhnpes4q541pwp80ZVnz0aVd4r479YplX+d584Blr8BD1Igc7kk29I61OKVL4eVm5mIeGa2eFr3GHZJTY4Jvlt2F09xxd0vJwTZudZkI0W1VS/LYuFUu0AAAAwAA9HGAtmGG4F0xWIM7vsBO4556g6qQ/teYAL6hK7JnxiPfJlVJGScFQaXFsioOIklICdwU3d3SoMpHbIROk9wKuMeluVJzMgM9Zz0TMpjBoYFMbWBLjh1uGquhYEyYz1nPCHFO4oTY3lp2PZO5mMyNVIL8oNg7ed9KmwCgnW/+gZFBjxVGn2tAmqUcebcEtSwGlKJ6ea3z7OU2+Ih0sgME7bPviJUxYOBmZ6z0ocj2GeCnp9ZCfln4O7u2vxKppIagMsKrpEEoT4ABbar4b6M3iAAYh53nxxHSLbbiS9tjAJutlmJdhtNuJutAPyrnvJKnx8i0na4rOqDPOtZICcTkWa3xZEf3XrlK4VBSMKUYqnUe4CpS8DCyTFgSldr8t20KrxGM16SxbNxStqAF4afwpwQTdO12YTxvJyYfVefUTQa1MFdTk/IObBtN0TzNIGn+jnlHXwwfxlxTJxjd3eCbezArcIqy90eUZgSIivsHwNXGguAn/DfZqTaTcZiP9kt13DyOz6hbtZrZm/57ZQm/fUx9YLVNal/M8P+1/V3TalFkkEqRL+AtYX9qp3hBCi1OZzbPJCe33/eWciLiOSuw7h09AFUkMILZcLeraZWLXZY6XlKkYjnvhZCBYP+zbfn3c8TEkY4ZBvVWo/ACBu4fJ1k6l7pVjS5SQEVPJQ/pjYrGGLcLPrQ5DRCQTiKjvgKV+iDzs3MV3+EVIm10YtrZHqz1ImsDLXqff3yYAoeFeAxU1uYc/4Jdb05/m+vcuiI03/842teMjtPaDMXY7EMynOhYE+djqt2K5rQwOkU71xkfnbZfXzLkeyv9kX/ieINfM+J1ACVPE8/7OaEVXLhyTKZiTNEh710Nw6orUvXlOl/dc2S4fVUTE79bXJiQUiWMvyInYH9NIdzUjXeQUOYLGX0Loitj/cXFEmSV0T12ty3F+8ad0Au//H74RH6/1omfkOU/+SKCeNThjOq90vVkbcoXo+NfPj/q09Sms0jiExKOkCKiLCokZJwtbi8vheJSwJBpklp78Fj1xDgG7vq51dJyXZIxFs7erQ1LgAAACbQp1VWywonzrCQJ559ysaYCVgiPgqZsng4e1R88bQ0tFYPNJc16bil9QlCI/xXAm1h7ZwP79k4hNsEHTxTAaFJcpR787WUgVXE9PSocMtGR12IF59dZH7j38xz7bp8aRyLXVW1TuG1I3jURsjvCK3KS7aNlhu6O2SgHFcIrdHbaJmnhnTZyZuY5Xw8ficV3pGd7T7FjeDKBzD2OHnHXaqjgPFmXQB/sizH/+i/ladxf2z+nJNikNMTrYTnEo09NILG3VuemqRtSixZvP4OM181RHs1urhdUUFN/xDOCWo8iJnTc+92+ym+TsU2+7No7b9CZDL5dHh73PcUjRuOKu4tOcLewCPeG9NKI4PXhPata2b+8Cmh3+R8W03Ch0kXIaoRRQ2QgBzXPctAwKehWLC5TUlm12qsSsTePaubeXGBrkCRUzJCsYJY69+a+NGb9W+9KrUx+ESCEBeyuTw6BVC8PGnakvCrqZGE25OUpJNaaAEBUoyMUUOXURfNmHzDpNCaoftWPfg7n/ptasQuyexvtk1zPM76u/c3fVBD3voyJsDk1eFa+zi5RU4zRmN2JKQQH4+MOf8vGdiBw+ha1X8AbKSQXcH6oGdpUPDbYirAkQ7YYFZD41hChn2snqNRH7k3uimulwA69n+cI6MF2SNF5hxZM9QL+GITWsOlmiW9fxwwfeRsM0RSjEmWnYqGi9HFGFHeulCBCyrYAAAMAIEofJT12whVOOKZfgMYsBB3BzdjRjcvYC+VDwVcMmxsFa/BjG9c0xHgGIQR9BOjRoRFsN85UNq30bWDTi/RU2UK6UwxVQNy7wpZTrX9ShEgk3Q71LAA6KiWRz/WIQIHbmLfxioPtw4S/MRbREupnU3x0REDShgDGVlY4TPLv8DienEVdv2k7E0uNDCBfH7RVLLBA77WJO1PgX7ustxJTny2P2nOkg19o56/hPpX57B41U9+4LTyXUMboRoQliU7eeqOVf7Su21SQWoyKPFVt6WmBFg8EbPX2wW4FDq+1Voge/vtPfz6g4duIY98DDvHgI8b0VG7FrGsRiYMKqebkmyGI/sOpjERaxt

## 9. Wall condition: free, pinned, and clamped

Three container configurations, selected by `wall`. See design doc §subsubsec:wall.

- `:free` (default) --- no-flux walls and a free $90^\circ$ contact line, $\partial_r\eta=0$
  at $r=b$. Basis: zeros of $J_1$, plus the piston mode $k_0=0$. Conserves bath volume
  exactly and structurally: $\kappa_0=0$, so the one mode that carries volume can never be
  driven.
- `:pinned` --- $\eta(b)=0$ exactly by construction (Dirichlet basis: zeros of $J_0$, no
  piston mode). A **diagnostic, not physics**: pinning this way destroys volume conservation,
  creating bath volume from nothing (measured at about four times the droplet's own volume
  over a reference impact).
- `:clamped` --- the recommended pinned option. Keeps the volume-conserving `:free` basis and
  imposes $\eta(b)=0$ as a constraint carried by a Lagrange multiplier (the rim line force),
  eliminated in closed form each step. Holds pinning and volume conservation at roundoff
  simultaneously --- see the run below.

At the production bath radius $b=6$ used throughout this notebook the wall condition barely
moves the contact time or CoR. It becomes a large, non-monotone effect once the container is
much smaller (see the README) --- not exercised here.

In [13]:
walls = (:free, :pinned, :clamped)
tcs_w = Float64[]; cors_w = Float64[]; wallmaxs = Float64[]; volmaxs = Float64[]
for wall in walls
    pw = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=3, b=6.0, h0=3.0, nq=40, wall=wall)
    lv, dg, ph = run_simulation(pw; t_end=14.0, dt_init=1e-3)
    tt = [l.t for l in lv]
    push!(tcs_w, primary_contact_time(tt, ph))
    push!(cors_w, coefficient_of_restitution(tt, lv, ph))
    push!(wallmaxs, maximum(abs(sum(l.bath.a[m+1] * besselj0(pw.k[m+1] * pw.b) for m in 0:pw.M)) for l in lv))
    push!(volmaxs, maximum(abs(sum(l.bath.a[m+1] * (pw.b / pw.k[m+1]) * besselj1(pw.k[m+1] * pw.b)
                                    for m in 1:pw.M)) for l in lv))
end
wallnames = string.(walls)
display(svgbars([(label=wallnames[i], series=[(name="|eta(b)| max", value=wallmaxs[i], color="#d62728"),
                                               (name="|int eta r dr| max", value=volmaxs[i], color="#1f77b4")])
                  for i in eachindex(walls)];
                 ylabel="residual", title="Pinning vs volume residual, by wall condition"))
display(svgbars([(label=wallnames[i], series=[(value=tcs_w[i], color="#1f77b4")]) for i in eachindex(walls)];
                 ylabel="t_cont", title="Contact time by wall condition"))
display(svgbars([(label=wallnames[i], series=[(value=cors_w[i], color="#2ca02c")]) for i in eachindex(walls)];
                 ylabel="CoR", title="Coefficient of restitution by wall condition"))
println()
println("`:pinned` holds the pin at roundoff but the volume residual is O(1). `:clamped`")
println("holds BOTH at roundoff -- that is the whole point of the multiplier construction.")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='640' height='300' font-family='sans-serif' font-size='12'><rect width='640' height='300' fill='white'/><line x1='56' y1='244.0' x2='51' y2='244.0' stroke='black'/><text x='47' y='248.0' text-anchor='end'>0.0</text><line x1='56' y1='192.8' x2='51' y2='192.8' stroke='black'/><text x='47' y='196.8' text-anchor='end'>0.289</text><line x1='56' y1='141.6' x2='51' y2='141.6' stroke='black'/><text x='47' y='145.6' text-anchor='end'>0.577</text><line x1='56' y1='90.39999999999998' x2='51' y2='90.39999999999998' stroke='black'/><text x='47' y='94.39999999999998' text-anchor='end'>0.866</text><line x1='56' y1='39.19999999999999' x2='51' y2='39.19999999999999' stroke='black'/><text x='47' y='43.19999999999999' text-anchor='end'>1.15</text><line x1='56' y1='244.0' x2='606.4' y2='244.0' stroke='black'/><line x1='56' y1='244' x2='56' y2='28.0' stroke='black'/><rect x='92.69333333333334' y='196.0841900364221' width='48.92444444444445' height='47.915809963577914' fill='#d62728'/><text x='117.15555555555557' y='190.0841900364221' text-anchor='middle' font-size='10'>0.27</text><rect x='153.8488888888889' y='243.99999999999915' width='48.92444444444445' height='8.526512829121202e-13' fill='#1f77b4'/><text x='178.31111111111113' y='237.99999999999915' text-anchor='middle' font-size='10'>4.82e-15</text><text x='147.73333333333335' y='262' text-anchor='middle'>free</text><rect x='276.15999999999997' y='243.9999999999998' width='48.92444444444445' height='1.9895196601282805e-13' fill='#d62728'/><text x='300.6222222222222' y='237.9999999999998' text-anchor='middle' font-size='10'>1.12e-15</text><rect x='337.31555555555553' y='65.91304347826085' width='48.92444444444445' height='178.08695652173915' fill='#1f77b4'/><text x='361.77777777777777' y='59.913043478260846' text-anchor='middle' font-size='10'>1.0</text><text x='331.2' y='262' text-anchor='middle'>pinned</text><rect x='459.62666666666667' y='244.0' width='48.92444444444445' height='0.0' fill='#d62728'/><text x='484.0888888888889' y='238.0' text-anchor='middle' font-size='10'>6.57e-17</text><rect x='520.7822222222222' y='243.9999999999984' width='48.92444444444445' height='1.5916157281026244e-12' fill='#1f77b4'/><text x='545.2444444444444' y='237.9999999999984' text-anchor='middle' font-size='10'>8.9e-15</text><text x='514.6666666666666' y='262' text-anchor='middle'>clamped</text><line x1='450' y1='42.0' x2='475' y2='42.0' stroke='#d62728' stroke-width='6'/><text x='481' y='46.0'>|eta(b)| max</text><line x1='450' y1='56.0' x2='475' y2='56.0' stroke='#1f77b4' stroke-width='6'/><text x='481' y='60.0'>|int eta r dr| max</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>residual</text><text x='320.0' y='18' text-anchor='middle' font-size='14'>Pinning vs volume residual, by wall condition</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='640' height='300' font-family='sans-serif' font-size='12'><rect width='640' height='300' fill='white'/><line x1='56' y1='244.0' x2='51' y2='244.0' stroke='black'/><text x='47' y='248.0' text-anchor='end'>0.0</text><line x1='56' y1='192.8' x2='51' y2='192.8' stroke='black'/><text x='47' y='196.8' text-anchor='end'>1.11</text><line x1='56' y1='141.6' x2='51' y2='141.6' stroke='black'/><text x='47' y='145.6' text-anchor='end'>2.22</text><line x1='56' y1='90.39999999999998' x2='51' y2='90.39999999999998' stroke='black'/><text x='47' y='94.39999999999998' text-anchor='end'>3.34</text><line x1='56' y1='39.19999999999999' x2='51' y2='39.19999999999999' stroke='black'/><text x='47' y='43.19999999999999' text-anchor='end'>4.45</text><line x1='56' y1='244.0' x2='606.4' y2='244.0' stroke='black'/><line x1='56' y1='244' x2='56' y2='28.0' stroke='black'/><rect x='111.04000000000002' y='76.95931427136722' width='73.38666666666667' height='167.04068572863278' fill='#1f77b4'/><text x='147.73333333333335' y='70.95931427136722' text-anchor='middle' font-size='10'>3.63</text><text x='147.73333333333335' y='262' text-anchor='middle'>free</text><rect x='294.50666666666666' y='69.59513374262963' width='73.38666666666667' height='174.40486625737037' fill='#1f77b4'/><text x='331.2' y='63.59513374262963' text-anchor='middle' font-size='10'>3.79</text><text x='331.2' y='262' text-anchor='middle'>pinned</text><rect x='477.9733333333333' y='65.91304347826085' width='73.38666666666667' height='178.08695652173915' fill='#1f77b4'/><text x='514.6666666666666' y='59.913043478260846' text-anchor='middle' font-size='10'>3.87</text><text x='514.6666666666666' y='262' text-anchor='middle'>clamped</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>t_cont</text><text x='320.0' y='18' text-anchor='middle' font-size='14'>Contact time by wall condition</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='640' height='300' font-family='sans-serif' font-size='12'><rect width='640' height='300' fill='white'/><line x1='56' y1='244.0' x2='51' y2='244.0' stroke='black'/><text x='47' y='248.0' text-anchor='end'>0.0</text><line x1='56' y1='192.8' x2='51' y2='192.8' stroke='black'/><text x='47' y='196.8' text-anchor='end'>0.0874</text><line x1='56' y1='141.6' x2='51' y2='141.6' stroke='black'/><text x='47' y='145.6' text-anchor='end'>0.175</text><line x1='56' y1='90.39999999999998' x2='51' y2='90.39999999999998' stroke='black'/><text x='47' y='94.39999999999998' text-anchor='end'>0.262</text><line x1='56' y1='39.19999999999999' x2='51' y2='39.19999999999999' stroke='black'/><text x='47' y='43.19999999999999' text-anchor='end'>0.35</text><line x1='56' y1='244.0' x2='606.4' y2='244.0' stroke='black'/><line x1='56' y1='244' x2='56' y2='28.0' stroke='black'/><rect x='111.04000000000002' y='68.51167917656386' width='73.38666666666667' height='175.48832082343614' fill='#2ca02c'/><text x='147.73333333333335' y='62.51167917656386' text-anchor='middle' font-size='10'>0.3</text><text x='147.73333333333335' y='262' text-anchor='middle'>free</text><rect x='294.50666666666666' y='65.91304347826085' width='73.38666666666667' height='178.08695652173915' fill='#2ca02c'/><text x='331.2' y='59.913043478260846' text-anchor='middle' font-size='10'>0.304</text><text x='331.2' y='262' text-anchor='middle'>pinned</text><rect x='477.9733333333333' y='66.08916167228364' width='73.38666666666667' height='177.91083832771636' fill='#2ca02c'/><text x='514.6666666666666' y='60.08916167228364' text-anchor='middle' font-size='10'>0.304</text><text x='514.6666666666666' y='262' text-anchor='middle'>clamped</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>CoR</text><text x='320.0' y='18' text-anchor='middle' font-size='14'>Coefficient of restitution by wall condition</text></svg>")


`:pinned` holds the pin at roundoff but the volume residual is O(1). `:clamped`
holds BOTH at roundoff -- that is the whole point of the multiplier construction.


## 10. Profiling: memory and wall time per impact

One impact's marginal cost is what `scripts/sweep.jl`'s own worker-count ablation is built
on. Reproduced here at a smaller scale: wall time for one impact, and the *incremental*
memory of running a second one after a warm-up. The warm-up pays Julia's one-time
compilation cost; only the marginal RSS growth after that tells you the per-case budget a
sweep should plan around.

One caveat on reading the printed numbers: `Sys.maxrss()` is *peak RSS since the process started*, not current RSS. By this point in the notebook several full-resolution impacts have already run (the N-sweep, the wall-condition loop), so the peak may already sit near its ceiling before this cell's own "warm-up" even begins -- a near-zero marginal reading here reflects that, not an absence of per-case cost. `scripts/sweep.jl`'s own ablation measures this cleanly from a freshly started process; treat the numbers below as a demonstration of the method, not a replacement for that measurement.

In [14]:
run_simulation(p; t_end=1.0, dt_init=1e-3)  # warm-up: compiles everything, do not time this
GC.gc()
rss0 = Sys.maxrss()
t1 = @elapsed run_simulation(p; t_end=14.0, dt_init=1e-3)
rss1 = Sys.maxrss()
t2 = @elapsed run_simulation(p; t_end=14.0, dt_init=1e-3)
rss2 = Sys.maxrss()
@printf("wall time per impact: %.3f s (run 1), %.3f s (run 2)\n", t1, t2)
@printf("peak RSS after warm-up:        %.1f MiB\n", rss0 / 2^20)
@printf("marginal peak RSS, run 1:       %.1f MiB\n", (rss1 - rss0) / 2^20)
@printf("marginal peak RSS, run 2:       %.1f MiB\n", (rss2 - rss1) / 2^20)
println()
if rss1 == rss0 && rss2 == rss1
    println("Marginal peak RSS is 0 here because this notebook already ran several")
    println("full-resolution impacts above (the N-sweep, the wall loop), so peak RSS")
    println("had already plateaued before this cell started -- see the caveat above.")
end
println("This is the same measurement scripts/sweep.jl's ablation uses to cap the worker")
println("count by a memory budget rather than by Sys.CPU_THREADS alone.")

wall time per impact: 23.936 s (run 1), 24.067 s (run 2)
peak RSS after warm-up:        755.8 MiB


marginal peak RSS, run 1:       0.0 MiB
marginal peak RSS, run 2:       0.0 MiB

Marginal peak RSS is 0 here because this notebook already ran several
full-resolution impacts above (the N-sweep, the wall loop), so peak RSS
had already plateaued before this cell started -- see the caveat above.
This is the same measurement scripts/sweep.jl's ablation uses to cap the worker
count by a memory budget rather than by Sys.CPU_THREADS alone.


## 11. Sweeps, and validation

For many cases use `scripts/sweep.jl`, which runs cases concurrently at a worker count it
picks by measuring rather than guessing:

```
julia --project=. -t auto scripts/sweep.jl --wall=free 0.2 0.4 1.0958 3.0
```

It calibrates one case's marginal memory footprint (about 12 MiB, against a fixed ~450 MiB
runtime cost), caps the worker count by a memory budget, then time-ablates $W=1,2,4,\dots$
and takes the knee. On an 8-core machine the knee is typically $W=4$: going to 8 workers buys
about 6%, because these runs contend on memory bandwidth and the GC. The sweep is resumable
--- completed Weber numbers are skipped on a re-run.

**Validation is against experiment only.** `scripts/validate_experimental.jl` compares the
droplet top and bottom trajectories against measured data with digitised error bars. The
model sits at about 1.4x the experimental error-bar half-length. Scripts that compare
against the 1PKM model are named `compare_*`, not `validate_*`, deliberately: another
model's output is not evidence.

**Known open issues**, none hidden:

- The contact radius peaks at $\tau\approx0.89$ against DNS's $1.46$. Not a resolution
  effect; refining `N` moves it *away* from DNS, implicating the unresolved pressure profile.
  No experimental contact radius exists to adjudicate.
- Convergence in `M`, `L` and the spectral cutoff is **not** established. Only `N`
  insensitivity of the integrated metrics is checked (cell 6).
- The detachment chatter of cell 3 is a stepper artifact that has not been eliminated, only
  measured and routed around by reporting the primary interval.